# What Your PCA Loadings Are Really Telling You About the Molecule
### 6.3 — PCA I: Scores, Loadings, and Seeing Structure

A spectrum is hundreds of numbers, but it is not hundreds of *independent*
numbers — neighbouring channels rise and fall together, and a handful of
underlying chemical components drive the whole thing. **Principal Component
Analysis (PCA)** is the tool that finds those few underlying directions and lets
you *see* the structure in a set of spectra: which samples group together
(**scores**) and which spectral features drive the grouping (**loadings**).

This notebook is **not** "call `PCA()` and plot the scores." Most tutorials stop
at a pretty scores scatter. The thing that actually makes PCA useful to a
chemist is the other half — **reading the loadings as spectral features** — and
that is where we spend our effort. We build the reasoning:

- why spectra are **multivariate and redundant**, and what PCA does about it;
- PCA **from scratch** with the SVD (no scikit-learn), so the geometry is visible;
- how to read a **scree plot** and recover the true number of components;
- how to read a **scores plot** to see sample structure;
- **how to read a loading as a spectrum** — peak by peak, sign by sign — and map
  it back to real chemical bands (the heart of the lesson);
- and the crucial caveat: **PCA finds variance, not chemistry.** Loadings are
  only chemically meaningful once nuisance variation has been removed.

Because the data is simulated, we know the **true pure-component spectra** and
the **true concentration** behind every sample — the answer key that lets us
*prove* the scores track concentration and the loadings are built from real
bands, instead of just asserting it.

> **The one idea to hold onto.** A loading lives on the **same axis as your
> spectra**, so you read it the same way you read a spectrum: a feature at a
> given wavenumber means that channel matters to this component, and its **sign**
> tells you which samples score high there. PCA hands you these directions by
> chasing **variance** — so they line up with chemistry only when the biggest
> source of variance *is* chemistry. Remove the nuisances first (scatter,
> spikes, baseline), and the loadings turn into molecular information you can
> actually interpret.

## 1. Setup

The standard stack. We implement PCA **from scratch with NumPy's SVD** rather
than importing it, because the whole point of this lesson is to see what scores
and loadings *are* — not to hide them inside a black box. We reuse the project's
simulators only to manufacture data with known ground truth.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Build the multi-component set from the shared physics primitives; reuse the NIR
# simulator for the "scatter dominates" caution at the end (tying back to 4.3).
from simulated_data.core.axes import build_axis
from simulated_data.core.peaks import add_peaks
from simulated_data import nir

EXPORTS = Path("exports")
EXPORTS.mkdir(exist_ok=True)

SEED = 0                        # fix every random draw so your figures match ours
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

CLASS_COLORS = ["#1f77b4", "#ff7f0e", "#2ca02c"]    # one per sample class
COMP_COLORS = {"A": "#1f77b4", "B": "#ff7f0e", "C": "#2ca02c"}

## 2. Why "multivariate"? Because channels are not independent

Imagine measuring a peak height at one wavelength to track a component. If two
components both absorb there, that single number is ambiguous — and you have
thrown away the hundreds of *other* channels that could have resolved them. Real
spectra are **redundant**: a band is many adjacent channels moving together, and
a mixture's spectrum is a few component spectra added in varying amounts. So the
*true* dimensionality of a 400-channel data set might be just **three** — one
axis per chemical component.

PCA exploits exactly that. It rotates the hundreds of correlated channels into a
few new axes — **principal components** — ordered by how much of the data's
variance each one explains. The first few capture the real structure; the rest
are noise. Those new axes come with two readouts:

- **scores** — where each *sample* sits along the new axes (structure *between
  samples*);
- **loadings** — how much each *channel* contributes to each new axis (structure
  *across the spectrum*).

Scores tell you *which samples are alike*; loadings tell you *why*.

## 3. A three-component dataset with known chemistry

We construct 60 samples as mixtures of **three pure components** — A, B, and C —
each a small molecule with its own characteristic bands on a shared 600–1800
cm⁻¹ axis. Every sample is a non-negative blend ``c_A·A + c_B·B + c_C·C`` plus a
little detector noise. To create visible *structure*, the samples fall into three
classes, each **enriched** in one component, while all three concentrations also
vary independently from sample to sample (so all three chemical axes carry real
variance, not just two).

The pure spectra `P` and the concentration matrix `conc` are the **ground
truth**: later we check the scores against `conc` and the loadings against `P`.

In [ ]:
x = build_axis(600.0, 1800.0, n_points=400)          # shared Raman/IR-style axis (cm-1)

# Three pure components, each (center, FWHM, amplitude) bands. Distinct fingerprints
# with a little overlap -- realistic, and enough to make PCA do real work.
COMPONENTS = {
    "A": [(700.0, 35.0, 1.0), (1150.0, 45.0, 0.6)],
    "B": [(950.0, 40.0, 0.9), (1500.0, 55.0, 0.7)],
    "C": [(1300.0, 38.0, 1.0), (1650.0, 50.0, 0.7)],
}
P = np.vstack([add_peaks(x, bands) for bands in COMPONENTS.values()])   # (3, n_points)

# Concentrations: each class is enriched (+0.5) in one component, and every
# component also varies independently in [0.1, 1.0]. Result: three clusters that
# still span all three chemical directions.
rng = np.random.default_rng(SEED)
n_per_class = 20
labels = np.repeat([0, 1, 2], n_per_class)            # 0->A-rich, 1->B-rich, 2->C-rich
enrichment = np.zeros((3 * n_per_class, 3))
enrichment[np.arange(3 * n_per_class), labels] = 0.5
conc = enrichment + rng.uniform(0.1, 1.0, size=(3 * n_per_class, 3))   # (60, 3) TRUTH

# Beer-Lambert-style mixing: spectra are concentration-weighted sums of pure
# components, plus modest detector noise.
X_clean = conc @ P                                    # noise-free truth
X = X_clean + rng.normal(0, 0.01, X_clean.shape)      # what we "measure"

print("spectra X      :", X.shape)
print("concentrations :", conc.shape, "(columns = A, B, C; this is the answer key)")
print("classes        :", dict(zip(*np.unique(labels, return_counts=True))))

In [ ]:
# Look at the pure components first -- this is the chemistry the loadings should
# rediscover later, so keep these band positions in mind.
fig, ax = plt.subplots(figsize=(10, 4))
for (name, _), spec in zip(COMPONENTS.items(), P):
    ax.plot(x, spec, color=COMP_COLORS[name], lw=2, label=f"component {name}")
ax.set_xlabel("Raman shift (cm$^{-1}$)"); ax.set_ylabel("Intensity (a.u.)")
ax.set_title("The three pure components (the known chemistry)")
ax.legend()
fig.tight_layout(); fig.savefig(EXPORTS / "01_pure_components.png", dpi=150)
plt.show()

## 4. The raw data: high-dimensional and highly redundant

Overlay the 60 mixtures coloured by class — you can half-see the structure, but
it is tangled, because every spectrum is a blend. The **correlation heatmap** of
the channels makes the redundancy explicit: big blocks of channels move together
(they belong to the same band/component), which is precisely the redundancy PCA
will collapse into a few axes.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))

ax = axes[0]
for i in range(X.shape[0]):
    ax.plot(x, X[i], color=CLASS_COLORS[labels[i]], lw=0.6, alpha=0.7)
for k, name in enumerate("ABC"):
    ax.plot([], [], color=CLASS_COLORS[k], label=f"class {name}-rich")
ax.set_xlabel("Raman shift (cm$^{-1}$)"); ax.set_ylabel("Intensity (a.u.)")
ax.set_title("60 mixture spectra (coloured by class)"); ax.legend()

ax = axes[1]
# Correlation between channels: structure = blocks of co-varying wavenumbers.
C = np.corrcoef(X.T)
im = ax.imshow(C, extent=[x[0], x[-1], x[-1], x[0]], cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_title("Channel-to-channel correlation (the redundancy)")
ax.set_xlabel("Raman shift (cm$^{-1}$)"); ax.set_ylabel("Raman shift (cm$^{-1}$)")
fig.colorbar(im, ax=ax, fraction=0.046, label="correlation")
fig.tight_layout(); fig.savefig(EXPORTS / "02_raw_data.png", dpi=150)
plt.show()

Those red/blue blocks are bands of channels that rise and fall together. Four
hundred channels, but only a few independent patterns — PCA's job is to name
them.

## 5. Mean-centering: PCA's required first step

PCA describes how samples **vary around the average spectrum**, so we first
subtract that average from every spectrum. After centering, each channel has mean
zero and the data is expressed as *deviations* — which is what "directions of
maximum variance" should be measured from. (Whether to also **scale** each
channel is a separate decision, the subject of 6.2; here mean-centering is the
right and sufficient choice because all channels share the same intensity units.)

In [ ]:
mean_spectrum = X.mean(axis=0)          # the average spectrum, shape (n_points,)
Xc = X - mean_spectrum                  # mean-centered data: deviations from average

print("each centered channel has ~zero mean:", np.allclose(Xc.mean(axis=0), 0))

fig, ax = plt.subplots(figsize=(10, 3.6))
ax.plot(x, mean_spectrum, color="black", lw=2, label="mean spectrum (removed)")
for i in range(0, X.shape[0], 6):
    ax.plot(x, Xc[i], color=CLASS_COLORS[labels[i]], lw=0.7, alpha=0.8)
ax.axhline(0, color="k", lw=0.5)
ax.set_xlabel("Raman shift (cm$^{-1}$)"); ax.set_ylabel("Intensity (a.u.)")
ax.set_title("Mean-centering: PCA models the deviations (coloured) about the mean (black)")
ax.legend()
fig.tight_layout(); fig.savefig(EXPORTS / "03_mean_centering.png", dpi=150)
plt.show()

## 6. PCA from scratch with the SVD

Every PCA is one **singular value decomposition** of the mean-centered data:

$$X_c = U\,\Sigma\,V^{\mathsf T}$$

and the three things you want fall straight out of it:

- **loadings** are the rows of $V^{\mathsf T}$ — each a direction in spectrum-space
  (length `n_points`), the new axes themselves;
- **scores** are $U\Sigma$ — each sample's coordinates along those axes;
- the **variance explained** by component $i$ is $\sigma_i^2 / \sum_j \sigma_j^2$.

That is the entire algorithm. No iteration, no library — just one matrix
factorisation.

In [ ]:
def pca_svd(X):
    '''Principal Component Analysis via the SVD of the mean-centered data.

    Returns
    -------
    scores   : (n_samples, k)  sample coordinates on the components (U * Sigma)
    loadings : (k, n_points)   the component directions (rows of V^T)
    evr      : (k,)            fraction of total variance each component explains
    mean     : (n_points,)     the mean spectrum that was removed
    '''
    mean = X.mean(axis=0)
    Xc = X - mean
    U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
    scores = U * S                       # == U @ diag(S)
    loadings = Vt
    evr = S**2 / np.sum(S**2)
    return scores, loadings, evr, mean


scores, loadings, evr, mean_spectrum = pca_svd(X)
print("scores  :", scores.shape)
print("loadings:", loadings.shape)
print("variance explained (first 5 PCs):", np.round(evr[:5], 4))

## 7. The scree plot: how many real components?

Plot the variance each component explains. The data was built from **three**
pure components, so we expect three meaningful PCs and then a cliff down to the
noise floor. Reading the scree — "where does it flatten?" — is how you decide the
true dimensionality *without* knowing the answer in advance.

In [ ]:
k_show = 8
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(np.arange(1, k_show + 1), evr[:k_show] * 100, color="#4c72b0")
ax.plot(np.arange(1, k_show + 1), np.cumsum(evr[:k_show]) * 100, "o-",
        color="#dd8452", label="cumulative %")
ax.set_xlabel("Principal component"); ax.set_ylabel("Variance explained (%)")
ax.set_title("Scree plot: three components, then a cliff to the noise floor")
ax.legend()
fig.tight_layout(); fig.savefig(EXPORTS / "04_scree.png", dpi=150)
plt.show()

print("variance explained : PC1 %.1f%%  PC2 %.1f%%  PC3 %.1f%%  PC4 %.2f%%"
      % (evr[0]*100, evr[1]*100, evr[2]*100, evr[3]*100))
print("first three PCs capture %.1f%% of all variance" % (evr[:3].sum()*100))

PC1–PC3 hold ~99 % of the variance and PC4 drops to a fraction of a percent —
the elbow lands exactly at **three**, matching the three components we built in.
The remaining PCs are fitting noise.

## 8. The scores plot: seeing structure between samples

Scores place each sample in the new low-dimensional space. Plot PC1 vs PC2 and
the three classes separate into clusters — PCA found the grouping **without ever
being told the labels** (it is unsupervised). Because we know the true
concentrations, we can go further than admiring clusters and *prove* the scores
encode chemistry: regress each component's true concentration on the first three
scores.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 6))
for k, name in enumerate("ABC"):
    m = labels == k
    ax.scatter(scores[m, 0], scores[m, 1], c=CLASS_COLORS[k], s=55,
               edgecolor="white", label=f"class {name}-rich")
ax.axhline(0, color="k", lw=0.5); ax.axvline(0, color="k", lw=0.5)
ax.set_xlabel(f"PC1 scores ({evr[0]*100:.0f}%)")
ax.set_ylabel(f"PC2 scores ({evr[1]*100:.0f}%)")
ax.set_title("Scores: the three classes separate (unsupervised)")
ax.legend()
fig.tight_layout(); fig.savefig(EXPORTS / "05_scores.png", dpi=150)
plt.show()

In [ ]:
# Grade scores against the KNOWN concentrations. Two views:
#  (1) the single best PC for each component (which axis it lives on);
#  (2) the multiple R using PC1-3 together (do the 3 PCs span the chemistry?).
design = np.column_stack([np.ones(len(scores)), scores[:, :3]])
print("component | best single PC (corr) | multiple R on PC1-3")
for j, name in enumerate("ABC"):
    single = [np.corrcoef(scores[:, p], conc[:, j])[0, 1] for p in range(3)]
    best = int(np.argmax(np.abs(single)))
    beta, *_ = np.linalg.lstsq(design, conc[:, j], rcond=None)
    pred = design @ beta
    R = np.sqrt(1 - np.sum((conc[:, j]-pred)**2) / np.sum((conc[:, j]-conc[:, j].mean())**2))
    print("    %s     |   PC%d  (%+.2f)        |     %.3f" % (name, best+1, single[best], R))

Each component's concentration is recovered from the first three scores with a
multiple R of essentially **1.0** — the three PCs fully span the chemical space.
Notice PC1 carries a *contrast*: component B correlates **positively** with PC1
while A correlates **negatively**. That sign pattern is a preview of the
loadings, which we read next.

## 9. Reading a loading as a spectrum (the part that matters)

Here is the step most tutorials skip. A loading is **not** an abstract math
object — it is a vector of length `n_points`, defined on the **same wavenumber
axis as your spectra**. So you read it like a spectrum, with three rules:

1. **A peak in a loading marks a wavenumber that drives that component.** Where
   the loading is flat (near zero), that channel does not matter to this PC.
2. **The sign says which samples score high.** A *positive* loading peak means
   samples with more intensity there get *positive* scores on this PC; a
   *negative* peak means more intensity there pushes scores *negative*.
3. **Opposite signs in one loading mean a trade-off.** Two bands with opposite
   sign tell you those features *anti-correlate* across your samples — as one
   goes up, the other tends to go down.

Plot the first three loadings beneath the pure components and read them off.

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(10, 9), sharex=True)

# Top: the pure components as a reference (the chemistry).
ax = axes[0]
for (name, _), spec in zip(COMPONENTS.items(), P):
    ax.plot(x, spec, color=COMP_COLORS[name], lw=1.8, label=f"comp {name}")
ax.set_ylabel("pure\n(reference)"); ax.legend(ncol=3, fontsize=8); ax.set_title(
    "Loadings read against the known component bands")

# Mark each component's main band positions to compare against the loadings.
band_marks = [(700, "A"), (1150, "A"), (950, "B"), (1500, "B"), (1300, "C"), (1650, "C")]

for p in range(3):
    ax = axes[p + 1]
    ax.plot(x, loadings[p], color="#333", lw=1.4)
    ax.axhline(0, color="k", lw=0.5)
    for wn, name in band_marks:
        ax.axvline(wn, color=COMP_COLORS[name], ls=":", lw=1, alpha=0.7)
    ax.set_ylabel(f"PC{p+1}\n({evr[p]*100:.0f}%)")
axes[-1].set_xlabel("Raman shift (cm$^{-1}$)")
fig.tight_layout(); fig.savefig(EXPORTS / "06_loadings_as_chemistry.png", dpi=150)
plt.show()

In [ ]:
# Make the reading quantitative: the loading value at each component's MAIN band.
main_band = {"A": 700, "B": 950, "C": 1300}
idx = {n: int(np.argmin(np.abs(x - wn))) for n, wn in main_band.items()}
print("loading value at each component's main band (sign = direction):")
print("        A(700)   B(950)   C(1300)")
for p in range(3):
    print("  PC%d  %+7.3f  %+7.3f  %+7.3f"
          % (p+1, loadings[p][idx['A']], loadings[p][idx['B']], loadings[p][idx['C']]))

Read straight off the numbers and the figure:

- **PC1** has a **positive** peak on component **B**'s bands and a **negative**
  peak on component **A**'s bands. So PC1 is an **A-versus-B contrast axis**:
  B-rich samples score high (positive), A-rich samples score low (negative) —
  exactly the signs we saw in the scores grading.
- **PC2** is dominated by a strong **negative** peak on component **C**'s bands:
  it is the **C axis**, with C-rich samples scoring low.
- **PC3** carries the smaller remaining contrasts.

The loading peaks land **on the real component bands**, not between them. We can
confirm there is no leftover, uninterpretable structure: regress each loading
onto the pure-component spectra and almost all of it is explained.

To make the reading unmistakable, here is the **PC1 loading on its own**,
annotated. The recipe is the whole skill: find the peaks, note their sign, name
the band. PC1 dips at A's bands and rises at B's bands — so a high PC1 score
*reads* as "more B, less A."

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.3))
ax.plot(x, loadings[0], color="#333", lw=1.6)
ax.axhline(0, color="k", lw=0.6)
ax.fill_between(x, loadings[0], 0, where=loadings[0] > 0, color="#ff7f0e", alpha=0.25)
ax.fill_between(x, loadings[0], 0, where=loadings[0] < 0, color="#1f77b4", alpha=0.25)

# Annotate the two A bands (negative) and two B bands (positive).
notes = [(700, "A band\n(negative)", "#1f77b4"), (1150, "A band\n(negative)", "#1f77b4"),
         (950, "B band\n(positive)", "#ff7f0e"), (1500, "B band\n(positive)", "#ff7f0e")]
for wn, label, color in notes:
    i = int(np.argmin(np.abs(x - wn)))
    ax.annotate(label, (x[i], loadings[0][i]), textcoords="offset points",
                xytext=(0, 22 if loadings[0][i] > 0 else -34), ha="center", fontsize=8,
                color=color, arrowprops=dict(arrowstyle="->", color=color))
ax.set_xlabel("Raman shift (cm$^{-1}$)"); ax.set_ylabel("PC1 loading weight")
ax.set_title("Reading the PC1 loading: positive = B bands, negative = A bands  →  PC1 is a B-vs-A axis")
fig.tight_layout(); fig.savefig(EXPORTS / "07_reading_a_loading.png", dpi=150)
plt.show()

In [ ]:
# Are the loadings entirely built from the real component bands? Regress each
# loading onto the (centered) pure spectra and report R^2. ~1.0 => the loading is
# a pure combination of real chemistry, with nothing unexplained.
Pc = P - P.mean(axis=1, keepdims=True)
print("how much of each loading is explained by the real component bands:")
for p in range(3):
    target = loadings[p] - loadings[p].mean()
    coef, *_ = np.linalg.lstsq(Pc.T, target, rcond=None)
    recon = Pc.T @ coef
    r2 = 1 - np.sum((target - recon)**2) / np.sum(target**2)
    print("  PC%d : R^2 = %.3f   (A,B,C weights = %s)"
          % (p+1, r2, np.array2string(coef, precision=2, sign='+')))

R² ≈ 1.0 for every component: the loadings are **linear combinations of the true
component spectra** and nothing else. That is what "loadings *are* chemistry"
means concretely — and it is true here precisely because we removed all the
nuisance variation by construction. Section 11 shows what happens when we don't.

## 10. Reconstruction: how PCA compresses, and how to confirm you kept enough

Scores and loadings together **rebuild** the spectra:
$\hat X = \text{mean} + \text{scores}_{[:k]}\,\text{loadings}_{[:k]}$. Keep only
the first `k` components and you compress 400 channels into `k` numbers per
sample. Compare the reconstruction to the **noise-free truth** as `k` grows: the
error should fall fast, hit the noise floor at `k = 3`, and then stop improving
(adding PC4+ just refits noise).

In [ ]:
def reconstruct(scores, loadings, mean, k):
    '''Rebuild the spectra from the first k principal components.'''
    return mean + scores[:, :k] @ loadings[:k]

print("k components | RMSE vs noise-free truth")
for k in [1, 2, 3, 4, 10]:
    Xk = reconstruct(scores, loadings, mean_spectrum, k)
    rmse = np.sqrt(np.mean((Xk - X_clean) ** 2))
    print("     %2d      |   %.4f" % (k, rmse))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.3))
ks = np.arange(1, 11)
rmses = [np.sqrt(np.mean((reconstruct(scores, loadings, mean_spectrum, k) - X_clean)**2))
         for k in ks]
axes[0].plot(ks, rmses, "o-", color="#4c72b0")
axes[0].axvline(3, color="red", ls="--", label="true # components")
axes[0].set_xlabel("components kept (k)"); axes[0].set_ylabel("RMSE vs truth")
axes[0].set_title("Error collapses at k = 3, then flattens"); axes[0].legend()

ex = 0
axes[1].plot(x, X[ex], color="#bbb", lw=2, label="measured (noisy)")
for k in [1, 3]:
    axes[1].plot(x, reconstruct(scores, loadings, mean_spectrum, k)[ex], lw=1.4,
                 label=f"reconstruction k={k}")
axes[1].set_xlabel("Raman shift (cm$^{-1}$)"); axes[1].set_ylabel("Intensity (a.u.)")
axes[1].set_title("One sample: k=1 misses bands, k=3 nails it"); axes[1].legend()
fig.tight_layout(); fig.savefig(EXPORTS / "08_reconstruction.png", dpi=150)
plt.show()

The error bottoms out at exactly three components and a single spectrum is
rebuilt faithfully from just **three numbers** plus the shared loadings — that is
the compression PCA buys you, and the reconstruction error is a practical way to
confirm you kept enough components on real data.

## 11. The caution: PCA finds **variance**, not chemistry

Everything above worked because the only thing varying between our samples *was*
chemistry. PCA does not know what chemistry is — it chases **variance**. If the
biggest source of variance is a **nuisance**, then PC1 will be that nuisance and
your loadings will be uninterpretable garbage dressed up as a component.

To make the point with curriculum continuity, we reuse the **scatter-dominated
NIR set from 4.3**: related samples whose real property (an analyte band) is
buried under per-sample physical **scatter** (multiplicative slope + additive
offset). Run PCA on the raw spectra, then on the **SNV-corrected** spectra, and
watch what PC1 *means* change.

In [ ]:
def make_nir_set(n_samples=36, seed=1):
    '''Rebuild a 4.3-style NIR set: one diagnostic analyte band encodes the
    property, every sample gets its own scatter, recorded as ground truth.'''
    rng = np.random.default_rng(seed)
    prop = np.linspace(0.2, 0.9, n_samples)
    prop = prop[rng.permutation(n_samples)]                 # property, shuffled
    matrix = [(1450.0, 130.0, 0.70), (1930.0, 150.0, 0.90), (2250.0, 200.0, 0.55)]
    rows = []
    for p in prop:
        peaks = matrix + [(1720.0, 110.0, 0.15 + 0.55 * p)]   # analyte band height = property
        ds = nir.simulate(n_samples=1, peaks=peaks, concentration=1.0,
                          scatter=True, noise=0.003, seed=rng, n_points=700)
        rows.append(ds.X[0])
    return ds.x, np.vstack(rows), prop


def snv(X):
    '''Standard Normal Variate: per-spectrum center and scale (from 4.3).'''
    return (X - X.mean(axis=1, keepdims=True)) / X.std(axis=1, keepdims=True)


wl, Xnir, prop = make_nir_set()
scatter_level = Xnir.mean(axis=1)            # a stand-in for each sample's scatter offset

for tag, data in [("RAW (scatter in)", Xnir), ("SNV (scatter removed)", snv(Xnir))]:
    s, _, e, _ = pca_svd(data)
    r_prop = abs(np.corrcoef(s[:, 0], prop)[0, 1])
    r_scat = abs(np.corrcoef(s[:, 0], scatter_level)[0, 1])
    print("%-22s PC1=%.0f%% | PC1 vs PROPERTY r=%.2f | PC1 vs SCATTER r=%.2f"
          % (tag, e[0]*100, r_prop, r_scat))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
for ax, (tag, data) in zip(axes, [("RAW: PC1 is scatter", Xnir),
                                   ("SNV: PC1 is the property", snv(Xnir))]):
    s, _, e, _ = pca_svd(data)
    scat = Xnir.mean(axis=1)
    sp = ax.scatter(s[:, 0], s[:, 1], c=prop, cmap="viridis", s=55, edgecolor="white")
    ax.set_xlabel(f"PC1 ({e[0]*100:.0f}%)"); ax.set_ylabel(f"PC2 ({e[1]*100:.0f}%)")
    ax.set_title(tag)
    fig.colorbar(sp, ax=ax, fraction=0.046, label="true property")
fig.tight_layout(); fig.savefig(EXPORTS / "09_scatter_caution.png", dpi=150)
plt.show()

On the **raw** NIR data, PC1 holds ~94 % of the variance and correlates almost
perfectly with the **scatter** (r ≈ 1.0) while barely tracking the property
(r ≈ 0.3) — in the left scores plot the colour (the real property) is scrambled
along PC1, because PC1 *is the nuisance*. Apply **SNV** to remove the scatter
first and PC1 flips to correlate ~1.0 with the **property**: same algorithm, but
now the dominant variance is chemistry, so the loadings would be chemically
meaningful. **The lesson: PCA is only as good as the preprocessing in front of
it.** Garbage variance in, garbage components out.

## 12. A small reusable PCA helper

`pca_svd()` from Section 6 is already the reusable tool — it returns scores,
loadings, variance explained, and the mean. Here it is once more with a tidy
variance summary, ready to carry into 6.4 (diagnostics), 6.5 (PLS), and the
classification lessons.

In [ ]:
def pca_report(X, n_show=5):
    '''Run PCA and print a compact variance summary; return the full results.'''
    scores, loadings, evr, mean = pca_svd(X)
    table = pd.DataFrame({
        "PC": np.arange(1, n_show + 1),
        "variance_%": np.round(evr[:n_show] * 100, 2),
        "cumulative_%": np.round(np.cumsum(evr[:n_show]) * 100, 2),
    })
    print(table.to_string(index=False))
    return scores, loadings, evr, mean


_ = pca_report(X)

## Key Takeaways

- **Spectra are redundant.** A few chemical components drive hundreds of
  correlated channels; PCA finds those few directions.
- **Scores and loadings are two views of one decomposition.** Scores place the
  *samples* (structure between samples); loadings weight the *channels* (which
  spectral features drive each component).
- **Read a loading like a spectrum.** It lives on the wavenumber axis: peaks mark
  the channels that matter, signs say which samples score high, and opposite
  signs mean two features trade off. Map the peaks back to known bands.
- **The scree plot reveals dimensionality.** The elbow — here at three — tells
  you how many real components there are before the noise floor.
- **PCA is unsupervised and chases variance, not chemistry.** If a nuisance
  (scatter, baseline, spikes) dominates the variance, it dominates PC1, and the
  loadings are meaningless. Preprocess first (4.3, 4.5), *then* PCA.
- **Reconstruction confirms how many components to keep.** Error vs `k` falls to
  the noise floor at the true dimensionality and then flattens.
- **Ground truth makes it provable.** Because the data is simulated, the scores
  were graded against true concentrations and the loadings against true
  component spectra — not just eyeballed.

## Practical Checklist

1. **Preprocess before PCA.** Remove scatter/baseline/spikes and decide on
   scaling (6.2) — PCA will faithfully model whatever variance you leave in.
2. **Mean-center** (always) and choose scaling deliberately.
3. **Look at the scree plot** and keep the components before the elbow; ignore
   the noise-floor tail.
4. **Plot scores** (PC1–PC2, and PC3) to see grouping; colour by any metadata you
   have to interpret the axes.
5. **Always plot the loadings**, on the wavenumber axis, and identify the bands
   driving each PC. A PC you cannot interpret is a flag, not a result.
6. **Check reconstruction error vs k** to confirm you kept enough components.
7. **Sanity-check against known chemistry** where possible (spiked samples,
   standards) — the same way we used ground truth here.

## Common Mistakes

- **Stopping at the scores plot.** The scores show *that* samples differ; only
  the loadings tell you *why*. Skipping loadings throws away the chemistry.
- **Running PCA on un-preprocessed spectra.** Scatter or baseline becomes PC1,
  and every downstream interpretation is wrong.
- **Forgetting to mean-center** (or letting a tool center silently and not
  knowing it) — the first "component" then just points at the mean.
- **Over-reading tiny components.** PCs deep in the noise floor describe noise;
  interpreting their loadings invents chemistry.
- **Assuming PC1 = your property.** PC1 is the biggest *variance*, which is your
  property only if nothing else varies more.
- **Confusing scores and loadings.** Scores are per-sample, loadings are
  per-channel; plotting one where you meant the other is a classic slip.
- **Comparing scores across separate PCA runs.** Sign and scale of components are
  arbitrary per fit; don't compare raw scores between models.

## Reporting Guidance

- **State the preprocessing** (scatter correction, baseline, scaling,
  mean-centering) — PCA results are meaningless without it.
- **Report how many components** you kept and **why** (scree elbow, cumulative
  variance, reconstruction error), with the variance-explained per PC.
- **Show the loadings**, not just the scores, and **interpret the bands** that
  drive each component in chemical terms.
- **Label scores plots** with the variance each axis explains (e.g. "PC1, 46 %").
- **Note that component sign/scale are arbitrary**, so describe loadings by their
  band pattern, not by absolute sign alone.
- **Be explicit that PCA is unsupervised** — it found structure, it did not
  validate a hypothesis; quantitative prediction is PLS (6.5) with proper
  validation (6.6).

## Next Lesson

**6.4 — PCA II: Outlier Detection and Diagnostics.** Now that you can read scores
and loadings, the next step is trusting them: Hotelling's $T^2$ and $Q$-residuals
turn a scores plot into a principled outlier test — which samples are unusual
*within* the model, and which don't fit the model at all. From there, **6.5 —
PLS Regression** turns this unsupervised structure into quantitative prediction
of a property, with honest validation in **6.6**. The loadings-as-chemistry habit
you built here is what keeps those models interpretable instead of black boxes.